# 03 - Rolling-window training

Step 3 of `THESIS_GUIDE.md`. The six models are trained on the FinBench rolling protocol (4 years of training, 1 of validation, 1 of test; test years 2020-2024) in five universes (DJI, NASDAQ-100, S&P 500, EURO STOXX 50, STOXX Europe 600) and three horizons (T = 5 and L = 1, T = 20 and L = 5, T = 60 and L = 20, FinBench's daily, weekly and monthly settings). Sections 1-6 are at the main horizon L = 5; section 7 compares the horizons. Each model is trained **exactly as FinBench trains it** and keeps its weights with FinBench's own rule (decided 2026-10-06, `xaifin.training.trainer.SELECTION_RULES`); the data-quality filter is on.

This notebook reads the results store (`xaifin.training.rolling.collect_results`) and answers three questions:

1. **How well do the models predict?** IC, RankIC, ICIR and MSE per model, universe and test year.
2. **Are the numbers in FinBench's range?** FinBench publishes only MSE and MAE for these models (`results/analysis/finbench_table2.csv`).
3. **How weak is the signal?** The starting point of the thesis: if the predictions barely rank the stocks, explanations and portfolios built on them can be unstable.

Metrics are computed on the test year with the labels the model is scored on (the daily z-score of the forward return). **IC** and **RankIC** are the daily cross-sectional Pearson and Spearman correlations between predictions and labels, averaged over the test days; **ICIR** and **RankICIR** divide that mean by the standard deviation of the daily values. FactorVAE's label is a 20-day return (issue 3 in `docs/model_notes.md`), and FinFormer's loss ignores scale, so its MSE is not comparable (notebook 02).

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

from xaifin import viz
from xaifin.config import RESULTS_ROOT, UNIVERSE_NAMES, UNIVERSES
from xaifin.portfolio.backtest import expected_years
from xaifin.training.rolling import collect_results

SEQ_LEN, PRED_LEN = 20, 5
MODELS = viz.MODEL_ORDER
all_runs = collect_results()
runs = all_runs[(all_runs["seq_len"] == SEQ_LEN) & (all_runs["pred_len"] == PRED_LEN) & all_runs["universe"].isin(UNIVERSES)]
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print(f"{len(runs)} finished runs (T={SEQ_LEN}, L={PRED_LEN}), seeds {sorted(runs['seed'].unique())}")

414 finished runs (T=20, L=5), seeds [np.int64(0), np.int64(5), np.int64(42)]


## 1. The grid

One run per model, universe, test year and seed. EURO STOXX 50 and STOXX Europe 600 have no 2020 run: their membership history starts on 2020-09-30, so no stock is a member at the start of 2020 (notebook 01). Training time: the L = 5 runs on DJI, NASDAQ-100 and EURO STOXX 50 ran one at a time on an RTX 5090, the others up to eight at a time on an H200, so times are only comparable within a group; FactorVAE is the slow one (FinBench's 96 attention layers run one after another for every day).

In [2]:
expected = {u: expected_years(u) for u in UNIVERSES}
coverage = (runs.groupby(["model", "universe"]).size().unstack("universe").reindex(index=MODELS, columns=UNIVERSES)
            .fillna(0).astype(int))
coverage.loc["expected per model"] = [len(expected[u]) * runs["seed"].nunique() for u in UNIVERSES]
display(coverage)
timing = runs.groupby("model").agg(runs=("minutes", "size"), minutes_mean=("minutes", "mean"), minutes_total=("minutes", "sum"),
                                   epochs_run=("epochs_run", "mean"), kept_epoch=("kept_epoch", "mean")).reindex(MODELS)
timing

universe,dji,nasdaq100,sp500,sx5e,sxxp
model,,,,,
MASTER,15,15,15,12,12
MATCC,15,15,15,12,12
FactorVAE,15,15,15,12,12
HIST,15,15,15,12,12
DiscoverPLF,15,15,15,12,12
FinFormer,15,15,15,12,12
expected per model,15,15,15,12,12


,runs,minutes_mean,minutes_total,epochs_run,kept_epoch
model,,,,,
MASTER,69,9.7246,671.0000,19.0435,18.0435
MATCC,69,34.6493,2390.8000,70.0000,69.0000
FactorVAE,69,154.1696,10637.7000,30.0000,1.2464
HIST,69,29.4681,2033.3000,54.6087,23.6087
DiscoverPLF,69,52.8391,3645.9000,48.8406,31.3768
FinFormer,69,21.2667,1467.4000,17.0435,6.0435


## 2. Test performance per model, universe and year

Mean over the three seeds. Blue cells: the predictions rank the stocks in the right direction on average over the test year; red: the wrong direction.

In [3]:
cells = runs.groupby(["universe", "model", "test_year"])[["IC", "RankIC", "ICIR", "RankICIR", "MSE", "MAE"]].mean().reset_index()
for universe in UNIVERSES:
    grid = cells[cells["universe"] == universe]
    if grid.empty:
        continue
    viz.metric_heatmap(grid, "RankIC", f"{UNIVERSE_NAMES[universe]}: test RankIC by model and test year").show()
    display(grid.pivot(index="model", columns="test_year", values="RankIC").reindex([m for m in MODELS if m in set(grid["model"])]))

test_year,2020,2021,2022,2023,2024
model,,,,,
MASTER,0.0299,0.0194,-0.0155,0.0269,0.0146
MATCC,0.0313,0.0154,0.0161,0.0229,-0.0116
FactorVAE,0.0430,0.0870,-0.0583,0.0470,-0.0427
HIST,0.0463,0.0228,-0.0057,0.0376,0.0524
DiscoverPLF,0.0066,0.0201,0.0113,0.0501,0.0132
FinFormer,0.0320,0.0439,0.0205,0.0109,0.0161


test_year,2020,2021,2022,2023,2024
model,,,,,
MASTER,0.0272,0.0052,-0.0034,0.0154,0.0058
MATCC,0.0319,0.0100,-0.0159,0.0113,0.0115
FactorVAE,0.0022,-0.0732,-0.0772,0.0185,-0.0365
HIST,0.0237,0.0047,-0.0067,0.0028,0.0161
DiscoverPLF,0.0162,-0.0151,-0.0110,0.0067,0.0137
FinFormer,0.0449,-0.0042,-0.0291,-0.0029,-0.0241


test_year,2020,2021,2022,2023,2024
model,,,,,
MASTER,0.0258,-0.0110,-0.0030,0.0069,0.0054
MATCC,0.0241,0.0151,-0.0015,0.0097,0.0014
FactorVAE,0.0042,0.0415,-0.0079,0.0110,-0.0333
HIST,-0.0066,0.0171,0.0140,0.0209,-0.0163
DiscoverPLF,0.0092,-0.0080,-0.0020,0.0227,-0.0167
FinFormer,0.0115,-0.0048,0.0029,0.0009,-0.0054


test_year,2021,2022,2023,2024
model,,,,
MASTER,0.0062,-0.0184,-0.0291,-0.0195
MATCC,-0.0107,-0.0260,0.0051,-0.0115
FactorVAE,0.0481,0.0054,-0.0344,-0.0555
HIST,-0.0139,0.0089,0.0042,-0.0263
DiscoverPLF,-0.0015,0.0032,0.0086,-0.0377
FinFormer,0.0174,-0.0005,0.0099,-0.0029


test_year,2021,2022,2023,2024
model,,,,
MASTER,0.0075,0.0066,-0.0139,-0.0057
MATCC,0.0056,0.0220,-0.0018,-0.0022
FactorVAE,-0.0277,0.0063,-0.0182,-0.0277
HIST,-0.0032,-0.0088,-0.0035,0.0039
DiscoverPLF,-0.0052,0.0035,-0.0033,-0.0085
FinFormer,-0.0014,-0.0051,0.0061,0.0071


In [4]:
for universe in UNIVERSES:
    grid = cells[cells["universe"] == universe]
    if not grid.empty:
        viz.metric_by_year_chart(grid, "IC", f"{UNIVERSE_NAMES[universe]}: test IC per year").show()

## 3. Averages over the test years, and FinBench's numbers

Per model and universe: the mean over test years (and seeds) of each metric, and its standard deviation across years. FinBench's Table 2 reports MSE and MAE (for DJI, NASDAQ-100 and EURO STOXX 50 only) averaged over its rolling test years and seeds; its FactorVAE numbers come from the leaky readout (issue 1), and its MASTER, MATCC and FinFormer runs were not seeded (issue 8).

In [5]:
finbench = pd.read_csv(RESULTS_ROOT / "analysis" / "finbench_table2.csv").query("pred_len == @PRED_LEN")
summary = (runs.groupby(["universe", "model"])
           .agg(years=("test_year", "nunique"), IC=("IC", "mean"), IC_std=("IC", "std"), RankIC=("RankIC", "mean"),
                RankIC_std=("RankIC", "std"), ICIR=("ICIR", "mean"), RankICIR=("RankICIR", "mean"),
                MSE=("MSE", "mean"), MAE=("MAE", "mean"))
           .reset_index()
           .merge(finbench.rename(columns={"MSE": "FinBench MSE", "MAE": "FinBench MAE"})[["universe", "model", "FinBench MSE", "FinBench MAE"]],
                  on=["universe", "model"], how="left"))
summary["model"] = pd.Categorical(summary["model"], MODELS)
summary = summary.sort_values(["universe", "model"]).reset_index(drop=True)
summary

,universe,model,years,IC,IC_std,RankIC,RankIC_std,ICIR,RankICIR,MSE,MAE,FinBench MSE,FinBench MAE
0,dji,MASTER,5,0.0160,0.0275,0.0150,0.0243,0.0715,0.0673,0.9801,0.7464,1.2490,0.8590
1,dji,MATCC,5,0.0125,0.0204,0.0148,0.0180,0.0542,0.0659,1.5011,0.9449,1.1300,0.8180
2,dji,FactorVAE,5,0.0190,0.0646,0.0152,0.0594,0.1080,0.0795,0.8884,0.7247,0.9710,0.7580
3,dji,HIST,5,0.0284,0.0270,0.0307,0.0257,0.1193,0.1357,0.9962,0.7523,0.9970,0.7470
4,dji,DiscoverPLF,5,0.0206,0.0224,0.0203,0.0183,0.0943,0.0963,1.1253,0.8037,1.2110,0.8300
5,dji,FinFormer,5,0.0197,0.0239,0.0247,0.0249,0.0889,0.1126,2.2609,1.1748,1.6570,0.9820
6,nasdaq100,MASTER,5,0.0078,0.0121,0.0100,0.0137,0.0504,0.0661,1.0149,0.7177,1.0170,0.7200
7,nasdaq100,MATCC,5,0.0053,0.0151,0.0098,0.0177,0.0363,0.0690,1.7342,0.9890,1.7130,0.9810
8,nasdaq100,FactorVAE,5,-0.0339,0.0407,-0.0332,0.0414,-0.1927,-0.1798,0.9033,0.7184,0.9180,0.7250
9,nasdaq100,HIST,5,0.0049,0.0165,0.0081,0.0129,0.0246,0.0437,1.0339,0.7223,1.0300,0.7210


## 4. How weak is the signal?

Three views of the same question. (a) The share of runs whose test RankIC is above 0: a model with no skill lands there half of the time. (b) The t-statistic of the mean daily RankIC. The labels of consecutive days overlap (each is a 5-day return), so the daily values are correlated and RankICIR × √(test days) would overstate it; RankICIR × √(test days / L) counts only non-overlapping periods. About 2 or more means the year's mean is unlikely to be noise. (c) The best test RankIC per universe and year: the ceiling of what any of the six reaches.

In [6]:
days = 252  # test days per year
weak = runs.assign(t_stat=runs["RankICIR"] * np.sqrt(days / PRED_LEN))
share = (weak.groupby("model").agg(runs=("RankIC", "size"), positive_RankIC=("RankIC", lambda s: (s > 0).mean()),
                                   mean_RankIC=("RankIC", "mean"), significant=("t_stat", lambda s: (s.abs() >= 2).mean()))
         .reindex(MODELS))
display(share.style.format({"positive_RankIC": "{:.0%}", "significant": "{:.0%}", "mean_RankIC": "{:+.4f}"}))
best = weak.loc[weak.groupby(["universe", "test_year"])["RankIC"].idxmax(), ["universe", "test_year", "model", "RankIC", "t_stat"]]
best

,runs,positive_RankIC,mean_RankIC,significant
model,,,,
MASTER,69,59%,+0.0036,0%
MATCC,69,64%,+0.0066,0%
FactorVAE,69,41%,-0.0078,30%
HIST,69,57%,+0.0080,1%
DiscoverPLF,69,58%,+0.0033,1%
FinFormer,69,64%,+0.0062,4%


,universe,test_year,model,RankIC,t_stat
833,dji,2020,MASTER,0.0529,1.5343
213,dji,2021,FactorVAE,0.0963,3.2858
416,dji,2022,FinFormer,0.0340,1.0602
215,dji,2023,FactorVAE,0.0711,2.8780
630,dji,2024,HIST,0.0705,2.4835
469,nasdaq100,2020,FinFormer,0.0523,2.0003
672,nasdaq100,2021,HIST,0.0163,0.5375
880,nasdaq100,2022,MASTER,0.0102,0.3001
886,nasdaq100,2023,MASTER,0.0274,1.2987
49,nasdaq100,2024,DiscoverPLF,0.0261,1.2530


## 5. Seeds: same model, different training luck

Each cell (model, universe, test year) is trained with three seeds (0, 5, 42), which change only the random parts of training: the initial weights, the order of the training days, dropout. If the seeds of one model disagree as much as different models do, differences between models cannot be told apart from training luck. Per universe, averaged over test years: the spread across seeds within a model (standard deviation of RankIC, averaged over models) and the spread across models (standard deviation of the models' seed-averaged RankIC), and the share of cells where the seeds do not even agree on the sign of RankIC.

In [7]:
by_cell = runs.groupby(["universe", "test_year", "model"])["RankIC"]
cell = by_cell.agg(seed_mean="mean", seed_std="std", seeds="size",
                   sign_flip=lambda s: bool((s > 0).any() and (s < 0).any())).reset_index()
spread = (cell.groupby(["universe", "test_year"])
          .agg(within_model_seed_std=("seed_std", "mean"), across_model_std=("seed_mean", "std"),
               sign_disagreement=("sign_flip", "mean"))
          .groupby("universe").mean())
display(spread.style.format({"within_model_seed_std": "{:.4f}", "across_model_std": "{:.4f}", "sign_disagreement": "{:.0%}"}))
per_model = cell.groupby("model").agg(seed_std=("seed_std", "mean"), sign_disagreement=("sign_flip", "mean")).reindex(MODELS)
per_model.style.format({"seed_std": "{:.4f}", "sign_disagreement": "{:.0%}"})

,within_model_seed_std,across_model_std,sign_disagreement
universe,,,
dji,0.0148,0.0235,17%
nasdaq100,0.0094,0.0207,37%
sp500,0.0074,0.0125,33%
sx5e,0.0130,0.0190,42%
sxxp,0.0073,0.0111,58%


,seed_std,sign_disagreement
model,,
MASTER,0.0115,39%
MATCC,0.0099,35%
FactorVAE,0.0116,13%
HIST,0.0099,39%
DiscoverPLF,0.0089,35%
FinFormer,0.0106,57%


## 6. The selection rules

Each model keeps its weights with FinBench's rule. If the validation score that picks the weights carried information about the test year, the kept epoch's validation IC and the test IC would be related; with a weak signal they need not be. The table shows, per model, the correlation across runs between the validation IC of the kept epoch and the test IC, and how many epochs the rule ran and kept.

In [8]:
selection = (runs.groupby("model")
             .apply(lambda d: pd.Series({"runs": len(d), "valid_IC_mean": d["valid_IC"].mean(), "test_IC_mean": d["IC"].mean(),
                                         "corr(valid IC, test IC)": d["valid_IC"].corr(d["IC"]) if len(d) > 2 else np.nan,
                                         "epochs_run": d["epochs_run"].mean(), "kept_epoch": d["kept_epoch"].mean()}),
                    include_groups=False)
             .reindex(MODELS))
selection

,runs,valid_IC_mean,test_IC_mean,"corr(valid IC, test IC)",epochs_run,kept_epoch
model,,,,,,
MASTER,69.0000,0.0048,0.0034,-0.1916,19.0435,18.0435
MATCC,69.0000,0.0103,0.0048,-0.1006,70.0000,69.0000
FactorVAE,69.0000,0.0186,-0.0061,-0.1564,30.0000,1.2464
HIST,69.0000,0.0323,0.0070,0.1473,54.6087,23.6087
DiscoverPLF,69.0000,0.0147,0.0024,0.0651,48.8406,31.3768
FinFormer,69.0000,0.0235,0.0057,0.2213,17.0435,6.0435


## 7. Horizons: L = 1, 5, 20

The same models trained to predict the next day (T = 5, L = 1), week (T = 20, L = 5) and month (T = 60, L = 20) of returns, on all five universes. (a) Mean test RankIC per model, universe and horizon (over test years and seeds). (b) How weak the signal is at each horizon, as in section 4: the t-statistic counts only non-overlapping periods, RankICIR × √(test days / L). (c) Seeds against models, as in section 5. Longer horizons have fewer independent periods per year (12 months against 252 days), so a RankIC of the same size is less certain.

In [9]:
horizon_runs = all_runs[all_runs["universe"].isin(UNIVERSES)].copy()
horizon_runs["t_stat"] = horizon_runs["RankICIR"] * np.sqrt(days / horizon_runs["pred_len"])
display(horizon_runs.groupby(["seq_len", "pred_len", "universe"]).size().unstack("universe").reindex(columns=UNIVERSES))
rank_ic = (horizon_runs.pivot_table(index="model", columns=["universe", "pred_len"], values="RankIC", aggfunc="mean")
           .reindex(index=MODELS, columns=UNIVERSES, level=0))
display(rank_ic.style.format("{:+.4f}").background_gradient(cmap="RdBu", vmin=-0.04, vmax=0.04))
weak_h = (horizon_runs.groupby(["model", "pred_len"])
          .agg(runs=("RankIC", "size"), mean_RankIC=("RankIC", "mean"), positive_RankIC=("RankIC", lambda s: (s > 0).mean()),
               significant=("t_stat", lambda s: (s.abs() >= 2).mean()))
          .unstack("pred_len").reindex(MODELS))
display(weak_h.style.format("{:.0%}", subset=["positive_RankIC", "significant"]).format("{:+.4f}", subset=["mean_RankIC"]))
cell_h = (horizon_runs.groupby(["pred_len", "universe", "test_year", "model"])["RankIC"]
          .agg(seed_mean="mean", seed_std="std", sign_flip=lambda s: bool((s > 0).any() and (s < 0).any())).reset_index())
spread_h = (cell_h.groupby(["pred_len", "universe", "test_year"])
            .agg(within_model_seed_std=("seed_std", "mean"), across_model_std=("seed_mean", "std"),
                 sign_disagreement=("sign_flip", "mean"))
            .groupby(["pred_len", "universe"]).mean().unstack("pred_len"))
spread_h.reindex(UNIVERSES).style.format("{:.4f}").format("{:.0%}", subset=["sign_disagreement"])

,universe,dji,nasdaq100,sp500,sx5e,sxxp
seq_len,pred_len,,,,,
5,1,90,90,90,72,72
20,5,90,90,90,72,72
60,20,90,90,90,72,72


## 8. Saved summary

`results/analysis/metrics_summary.csv`: one row per run, every universe and horizon (cell, test metrics, kept epoch, epochs run, minutes, validation IC of the kept epoch). Later steps and the thesis tables read it.

In [10]:
out = RESULTS_ROOT / "analysis" / "metrics_summary.csv"
all_runs.drop(columns="path").sort_values(["model", "universe", "seq_len", "pred_len", "test_year", "seed"]).to_csv(out, index=False)
print(f"{len(all_runs)} runs -> {out}")

1242 runs -> D:\CodeProjects\master-thesis\code\results\analysis\metrics_summary.csv


## Observations

The complete grid: 6 models x 5 universes x test years 2020-2024 (the two European universes from 2021) x seeds 0, 5, 42 x three horizons, data-quality filter on. Sections 1-6 are at L = 5 (414 runs), section 7 compares the horizons.

1. **1242 runs, no failure.** 117 were trained on an RTX 5090 (seed 42 and a few runs of seeds 0 and 5 at L = 5), the other 1125 on an H200 with up to eight runs sharing the GPU, so the minutes per run of section 1 are not comparable between the two groups. FactorVAE is by far the slowest (154 minutes per run on average), MASTER the fastest (10). The RTX 5090 runs were trained twice, before and after the H200 grid: 99 of 117 are bit-identical; the 18 MATCC runs differ slightly (RankIC by 0.008 on average), because MATCC uses non-deterministic GPU kernels.
2. **The signal is weak everywhere.** At L = 5, each model's mean test RankIC over its 69 runs is between -0.008 (FactorVAE) and +0.008 (HIST). RankIC is above 0 in 41% (FactorVAE) to 64% (MATCC, FinFormer) of the runs, where a model with no skill would land at about 50%. Counting only non-overlapping weeks, a run's mean RankIC is clearly different from 0 (|t| >= 2) in at most 3 of 69 runs for every model except FactorVAE (30%, mostly negative: 9% positive).
3. **It depends on the universe more than on the model.** On DJI every model's average RankIC is positive (0.015 to 0.031); on the S&P 500 it is small and positive (0.001 to 0.010); on NASDAQ-100 it ranges from -0.033 (FactorVAE) to +0.010; on the two European universes most models are at or below 0 (EURO STOXX 50 -0.015 to +0.006, STOXX Europe 600 -0.017 to +0.006).
4. **No model wins consistently.** The best model of a universe and year changes from cell to cell (section 4, last table): FactorVAE, HIST and MATCC top 5 of the 23 cells each, MASTER 4, FinFormer 3, DiscoverPLF 1, and the best run of a cell reaches a RankIC between 0.009 and 0.096.
5. **Training luck is about half to two thirds of the model differences.** Within a universe and year, the RankIC of one model's three seeds varies by 0.007-0.015 (standard deviation, averaged over models), against 0.011-0.024 across the six models; in 17% (DJI) to 58% (STOXX Europe 600) of the cells the three seeds do not even agree on the sign of RankIC (section 5). FinFormer's seeds disagree most often (57% of its cells), FactorVAE's least (13%), because its rule keeps an almost untrained model.
6. **The selection rules choose almost blindly.** Across runs, the validation IC of the kept epoch and the test IC are barely related (correlation -0.19 to 0.22 by model), and validation scores are optimistic (0.005-0.032 against -0.006 to 0.007 on the test year). FactorVAE's rule (lowest validation loss) keeps epoch 1 of 30 on average.
7. **The horizon matters (section 7).** L = 1 carries the most signal: RankIC is positive in 68-84% of the runs of every model but FactorVAE (46%), and significantly so in a third of MASTER's runs and a quarter of MATCC's and FactorVAE's. At L = 20 the models predict nothing: the mean RankIC is at or below 0 for all but DiscoverPLF (0.001), positive in 42-54% of the runs, and almost never significant. The spread between seeds grows with the horizon (0.007 at L = 1, 0.010 at L = 5, 0.015 at L = 20), as does the spread between models, because a longer horizon leaves fewer independent periods in a test year. FactorVAE's label covers T rather than L days (5, 20 and 60 days), so its row is not on the same footing.

The regression errors against FinBench's Table 2 are compared, for the three horizons, by `scripts/compare_finbench.py` (chapter 4).

Starting point for the thesis: the six architectures extract a small, unstable signal whose sign changes with the period, the universe, the horizon and even the seed. Whether their explanations agree, and whether disagreement comes from the data or the models, is the question of Steps 5-7.